# Hierarchical Plant Disease Classifier

This notebook builds a hierarchical setup with separate disease models:
- `citrus_model`: trained only on citrus classes
- `mango_model`: trained only on mango classes

It follows the workflow style of `02. mobilenet_model.ipynb` with data loading, training, evaluation, saving, and inference.


## 1. Setup and Imports


In [1]:
import sys
from pathlib import Path

# Ensure project root is importable when running from notebooks/
project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.datasets import ImageFolder
from PIL import Image

from src.config import (
    TRAIN_DIR,
    VAL_DIR,
    BATCH_SIZE,
    PIN_MEMORY,
    MOBILENET_NUM_EPOCHS,
    MOBILENET_MODEL_SAVE_PATH,
    OVERSAMPLING_ENABLED,
    OVERSAMPLING_POWER,
    OVERSAMPLING_MIN_MULTIPLIER,
    OVERSAMPLING_MAX_MULTIPLIER,
    OVERSAMPLING_EPOCH_MULTIPLIER,
)
from src.models import get_mobilenet_model
from src.transforms import get_light_transform, get_strong_transform, get_val_transform
from src.train import setup_training_mobilenet, train_model

print('Imports complete')

# Notebook-local safety settings (Windows/Jupyter friendly)
HIER_NUM_WORKERS = 0
HIER_NUM_EPOCHS = min(8, MOBILENET_NUM_EPOCHS)


Imports complete


## 2. Device


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cuda


## 3. Load Base Datasets


In [3]:
train_base = ImageFolder(root=TRAIN_DIR, transform=None)
val_base = ImageFolder(root=VAL_DIR, transform=None)

all_classes = train_base.classes
print(f'Total classes: {len(all_classes)}')
print(all_classes)
print(f'Train samples: {len(train_base)} | Val samples: {len(val_base)}')


Total classes: 15
['citrus_black_spot', 'citrus_canker', 'citrus_foliage_damage', 'citrus_greening', 'citrus_healthy', 'citrus_mealybugs', 'citrus_melanose', 'mango_anthracnose', 'mango_bacterial_canker', 'mango_cutting_weevil', 'mango_die_back', 'mango_gall_midge', 'mango_healthy', 'mango_powdery_mildew', 'mango_sooty_mould']
Train samples: 24499 | Val samples: 6127


## 4. Family Split Helpers


In [4]:
def get_family_class_names(classes, family_prefix):
    return [c for c in classes if c.startswith(family_prefix)]


def get_family_indices(imagefolder_dataset, family_prefix):
    indices = []
    for i, (_, label) in enumerate(imagefolder_dataset.samples):
        cls_name = imagefolder_dataset.classes[label]
        if cls_name.startswith(family_prefix):
            indices.append(i)
    return indices


class FamilyMappedDataset(Dataset):
    def __init__(self, base_dataset, subset_indices, class_names, transform):
        self.base = base_dataset
        self.indices = subset_indices
        self.class_names = class_names
        self.transform = transform

        global_to_name = {i: n for i, n in enumerate(base_dataset.classes)}
        self.name_to_local = {n: i for i, n in enumerate(class_names)}

        self.local_labels = []
        for idx in self.indices:
            _, global_label = base_dataset.samples[idx]
            name = global_to_name[global_label]
            self.local_labels.append(self.name_to_local[name])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        base_idx = self.indices[i]
        path, global_label = self.base.samples[base_idx]
        img = self.base.loader(path)
        if self.transform is not None:
            img = self.transform(img)

        cls_name = self.base.classes[global_label]
        local_label = self.name_to_local[cls_name]
        return img, local_label


def create_family_sampler(local_labels, num_classes):
    class_counts = Counter(local_labels)
    class_weights = np.ones(num_classes, dtype=np.float64)

    if OVERSAMPLING_ENABLED:
        max_count = max(class_counts.values())
        for cls_idx in range(num_classes):
            count = class_counts.get(cls_idx, 1)
            mult = (max_count / count) ** OVERSAMPLING_POWER
            mult = np.clip(mult, OVERSAMPLING_MIN_MULTIPLIER, OVERSAMPLING_MAX_MULTIPLIER)
            class_weights[cls_idx] = mult

    sample_weights = torch.DoubleTensor([class_weights[y] for y in local_labels])
    num_samples = max(1, int(len(sample_weights) * OVERSAMPLING_EPOCH_MULTIPLIER))

    return WeightedRandomSampler(sample_weights, num_samples=num_samples, replacement=True), class_counts


def build_family_loaders(family_prefix):
    class_names = get_family_class_names(all_classes, family_prefix)
    train_idx = get_family_indices(train_base, family_prefix)
    val_idx = get_family_indices(val_base, family_prefix)

    train_ds = FamilyMappedDataset(train_base, train_idx, class_names, transform=get_light_transform())
    val_ds = FamilyMappedDataset(val_base, val_idx, class_names, transform=get_val_transform())

    sampler, class_counts = create_family_sampler(train_ds.local_labels, len(class_names))

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=HIER_NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=HIER_NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    return class_names, train_ds, val_ds, train_loader, val_loader, class_counts


## 5. Build Citrus and Mango Pipelines


In [5]:
citrus_classes, citrus_train_ds, citrus_val_ds, citrus_train_loader, citrus_val_loader, citrus_counts = build_family_loaders('citrus')
mango_classes, mango_train_ds, mango_val_ds, mango_train_loader, mango_val_loader, mango_counts = build_family_loaders('mango')

print('CITRUS classes:', citrus_classes)
print('MANGO classes :', mango_classes)
print(f'Citrus train/val: {len(citrus_train_ds)}/{len(citrus_val_ds)}')
print(f'Mango  train/val: {len(mango_train_ds)}/{len(mango_val_ds)}')


CITRUS classes: ['citrus_black_spot', 'citrus_canker', 'citrus_foliage_damage', 'citrus_greening', 'citrus_healthy', 'citrus_mealybugs', 'citrus_melanose']
MANGO classes : ['mango_anthracnose', 'mango_bacterial_canker', 'mango_cutting_weevil', 'mango_die_back', 'mango_gall_midge', 'mango_healthy', 'mango_powdery_mildew', 'mango_sooty_mould']
Citrus train/val: 21299/5327
Mango  train/val: 3200/800


## 6. Train Citrus Model


In [6]:
citrus_model = get_mobilenet_model(len(citrus_classes), version='v2', dropout=0.2).to(device)
citrus_criterion, citrus_optimizer = setup_training_mobilenet(
    citrus_model,
    train_labels=citrus_train_ds.local_labels,
    num_classes=len(citrus_classes),
    device=device,
)

citrus_history = train_model(
    citrus_model,
    citrus_train_loader,
    citrus_val_loader,
    citrus_criterion,
    citrus_optimizer,
    device,
    num_epochs=HIER_NUM_EPOCHS,
    model_save_path='../models/MobileNet_citrus_best.pth',
)


Epoch 1/8 | Train Loss: 0.4404 | Train Acc: 0.8440 | Val Loss: 0.1433 | Val Acc: 0.9564
Epoch 2/8 | Train Loss: 0.2421 | Train Acc: 0.9200 | Val Loss: 0.1141 | Val Acc: 0.9623
Epoch 3/8 | Train Loss: 0.2081 | Train Acc: 0.9242 | Val Loss: 0.1012 | Val Acc: 0.9643
Epoch 4/8 | Train Loss: 0.1986 | Train Acc: 0.9263 | Val Loss: 0.1096 | Val Acc: 0.9563
EarlyStopping counter: 1/5
Epoch 5/8 | Train Loss: 0.1964 | Train Acc: 0.9292 | Val Loss: 0.1262 | Val Acc: 0.9441
EarlyStopping counter: 2/5
Epoch 6/8 | Train Loss: 0.1771 | Train Acc: 0.9297 | Val Loss: 0.1002 | Val Acc: 0.9596
EarlyStopping counter: 3/5
Epoch 7/8 | Train Loss: 0.1709 | Train Acc: 0.9332 | Val Loss: 0.0915 | Val Acc: 0.9638
Epoch 8/8 | Train Loss: 0.1676 | Train Acc: 0.9311 | Val Loss: 0.0815 | Val Acc: 0.9700

Training completed! Best validation accuracy: 0.9700


## 7. Train Mango Model


In [7]:
mango_model = get_mobilenet_model(len(mango_classes), version='v2', dropout=0.2).to(device)
mango_criterion, mango_optimizer = setup_training_mobilenet(
    mango_model,
    train_labels=mango_train_ds.local_labels,
    num_classes=len(mango_classes),
    device=device,
)

mango_history = train_model(
    mango_model,
    mango_train_loader,
    mango_val_loader,
    mango_criterion,
    mango_optimizer,
    device,
    num_epochs=HIER_NUM_EPOCHS,
    model_save_path='../models/MobileNet_mango_best.pth',
)


Epoch 1/8 | Train Loss: 0.7140 | Train Acc: 0.8453 | Val Loss: 0.1939 | Val Acc: 0.9850
Epoch 2/8 | Train Loss: 0.1523 | Train Acc: 0.9869 | Val Loss: 0.0906 | Val Acc: 0.9912
Epoch 3/8 | Train Loss: 0.0945 | Train Acc: 0.9894 | Val Loss: 0.0611 | Val Acc: 0.9962
Epoch 4/8 | Train Loss: 0.0750 | Train Acc: 0.9903 | Val Loss: 0.0461 | Val Acc: 0.9962
Epoch 5/8 | Train Loss: 0.0540 | Train Acc: 0.9944 | Val Loss: 0.0364 | Val Acc: 0.9975
Epoch 6/8 | Train Loss: 0.0462 | Train Acc: 0.9953 | Val Loss: 0.0309 | Val Acc: 0.9975
Epoch 7/8 | Train Loss: 0.0386 | Train Acc: 0.9941 | Val Loss: 0.0259 | Val Acc: 0.9988
Epoch 8/8 | Train Loss: 0.0332 | Train Acc: 0.9972 | Val Loss: 0.0261 | Val Acc: 0.9975
EarlyStopping counter: 1/5

Training completed! Best validation accuracy: 0.9988


## 8. Evaluate Family Models


In [8]:
def evaluate_family(model, loader, class_names, device, title=''):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            pred = torch.argmax(logits, dim=1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    print(f'\n{title} Classification Report:')
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    print(f'{title} Confusion Matrix:')
    print(cm)

    return y_true, y_pred, cm


_ = evaluate_family(citrus_model, citrus_val_loader, citrus_classes, device, title='Citrus')
_ = evaluate_family(mango_model, mango_val_loader, mango_classes, device, title='Mango')



Citrus Classification Report:
                       precision    recall  f1-score   support

    citrus_black_spot       0.94      0.86      0.90        35
        citrus_canker       0.99      1.00      0.99      2250
citrus_foliage_damage       0.88      0.81      0.85       420
      citrus_greening       0.89      0.95      0.92        41
       citrus_healthy       1.00      0.98      0.99      1277
     citrus_mealybugs       0.90      0.94      0.92       784
      citrus_melanose       1.00      1.00      1.00       520

             accuracy                           0.97      5327
            macro avg       0.94      0.93      0.94      5327
         weighted avg       0.97      0.97      0.97      5327

Citrus Confusion Matrix:
[[  30    0    0    5    0    0    0]
 [   0 2245    0    0    5    0    0]
 [   0    0  342    0    0   78    0]
 [   2    0    0   39    0    0    0]
 [   0   23    0    0 1254    0    0]
 [   0    0   47    0    0  737    0]
 [   0    0    0    

## 9. Save Hierarchical Artifacts


In [9]:
torch.save(
    {
        'model_state': citrus_model.state_dict(),
        'idx_to_class': citrus_classes,
        'family': 'citrus',
    },
    '../models/MobileNet_citrus_best_with_meta.pth',
)

torch.save(
    {
        'model_state': mango_model.state_dict(),
        'idx_to_class': mango_classes,
        'family': 'mango',
    },
    '../models/MobileNet_mango_best_with_meta.pth',
)

hier_metadata = {
    'router': 'confidence-based dual-model routing',
    'citrus_model_path': '../models/MobileNet_citrus_best_with_meta.pth',
    'mango_model_path': '../models/MobileNet_mango_best_with_meta.pth',
    'citrus_classes': citrus_classes,
    'mango_classes': mango_classes,
}

with open('../models/hierarchical_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(hier_metadata, f, indent=2)

print('Saved citrus/mango checkpoints + hierarchical metadata')


Saved citrus/mango checkpoints + hierarchical metadata


## 10. Hierarchical Inference (Two Disease Models)


In [10]:
# Inference settings
TEMPERATURE = 1.0
CONF_THRESHOLD = 0.35
MARGIN_THRESHOLD = 0.10
ROUTER_GAP_THRESHOLD = 0.05  # if family scores too close => uncertain


def _tta_variants(pil_img):
    return [
        pil_img,
        pil_img.transpose(Image.FLIP_LEFT_RIGHT),
        pil_img.rotate(8),
        pil_img.rotate(-8),
    ]


def _predict_family_model(model, class_names, img, device, temperature=1.0, use_tta=True):
    model.eval()
    tfm = get_val_transform()

    with torch.no_grad():
        if use_tta:
            probs_list = []
            for v in _tta_variants(img):
                x = tfm(v).unsqueeze(0).to(device)
                logits = model(x)
                probs_list.append(F.softmax(logits / temperature, dim=1))
            probs = torch.mean(torch.cat(probs_list, dim=0), dim=0)
        else:
            x = tfm(img).unsqueeze(0).to(device)
            logits = model(x)
            probs = F.softmax(logits / temperature, dim=1).squeeze(0)

    top2_probs, top2_idx = torch.topk(probs, k=min(2, len(class_names)))
    top1_conf = float(top2_probs[0].item())
    top1_idx = int(top2_idx[0].item())
    top1_cls = class_names[top1_idx]

    if len(class_names) > 1:
        top2_conf = float(top2_probs[1].item())
        top2_cls = class_names[int(top2_idx[1].item())]
        margin = top1_conf - top2_conf
    else:
        top2_conf = 0.0
        top2_cls = None
        margin = top1_conf

    return {
        'top1_class': top1_cls,
        'top1_confidence': top1_conf,
        'top2_class': top2_cls,
        'top2_confidence': top2_conf,
        'margin': margin,
    }


def hierarchical_predict(image_path, use_tta=True):
    img = Image.open(image_path).convert('RGB')

    citrus_out = _predict_family_model(
        citrus_model, citrus_classes, img, device,
        temperature=TEMPERATURE, use_tta=use_tta
    )
    mango_out = _predict_family_model(
        mango_model, mango_classes, img, device,
        temperature=TEMPERATURE, use_tta=use_tta
    )

    citrus_score = citrus_out['top1_confidence']
    mango_score = mango_out['top1_confidence']

    if citrus_score >= mango_score:
        chosen_family = 'citrus'
        chosen = citrus_out
        other = mango_out
    else:
        chosen_family = 'mango'
        chosen = mango_out
        other = citrus_out

    router_gap = abs(citrus_score - mango_score)
    uncertain = (
        chosen['top1_confidence'] < CONF_THRESHOLD
        or chosen['margin'] < MARGIN_THRESHOLD
        or router_gap < ROUTER_GAP_THRESHOLD
    )

    return {
        'predicted_class': chosen['top1_class'],
        'confidence': chosen['top1_confidence'],
        'second_best_class_same_family': chosen['top2_class'],
        'margin_same_family': chosen['margin'],
        'chosen_family': chosen_family,
        'family_router_gap': router_gap,
        'decision': 'uncertain' if uncertain else 'accept',
        'uncertain': bool(uncertain),
        'citrus_top1': citrus_out,
        'mango_top1': mango_out,
        'tta_used': bool(use_tta),
    }

print('Hierarchical inference ready')


Hierarchical inference ready


## 11. Inference Example


In [11]:
# Example
test_image = r"C:\Users\loaim\OneDrive\Desktop\img2.jpeg"
result = hierarchical_predict(test_image, use_tta=True)
print(result)


{'predicted_class': 'mango_healthy', 'confidence': 0.6855353713035583, 'second_best_class_same_family': 'mango_bacterial_canker', 'margin_same_family': 0.49856024980545044, 'chosen_family': 'mango', 'family_router_gap': 0.18309718370437622, 'decision': 'accept', 'uncertain': False, 'citrus_top1': {'top1_class': 'citrus_mealybugs', 'top1_confidence': 0.5024381875991821, 'top2_class': 'citrus_melanose', 'top2_confidence': 0.475178599357605, 'margin': 0.02725958824157715}, 'mango_top1': {'top1_class': 'mango_healthy', 'top1_confidence': 0.6855353713035583, 'top2_class': 'mango_bacterial_canker', 'top2_confidence': 0.1869751214981079, 'margin': 0.49856024980545044}, 'tta_used': True}


## 12. Notes


In [ ]:
print('Use this notebook as the hierarchical pipeline baseline.')
print('If needed, add a dedicated binary citrus-vs-mango router model for stronger stage-1 routing.')
